# Gugak Stem Separation — EDA (Phase 2)

Audio-content EDA. **Source of truth = the manifest** (`manifests/*.parquet`); we never walk directories.

Phase 2 questions:
1. **Stem naming / strip rule** — validate `instrument` → `instrument_base` (multi-instrument stems).
2. **Sample rate** — confirm the 96 kHz (창작국악) set to resample.
3. **Listen** — do multi-instrument stems (피리 / 피리2 / 피리3) share a timbre? (manual check)
4. **Does summing stems give the master?** — residual test on 3 exemplar songs.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import IPython.display as ipd
import matplotlib.pyplot as plt

# Find repo root by walking up until we see the manifest (notebook cwd-agnostic).
def find_root(start: Path | None = None) -> Path:
    p = Path.cwd() if start is None else start
    for cand in [p, *p.parents]:
        if (cand / 'manifests' / 'songs.parquet').exists():
            return cand
    raise FileNotFoundError('repo root (with manifests/) not found above cwd')

ROOT = find_root()
songs = pd.read_parquet(ROOT / 'manifests' / 'songs.parquet')
stems = pd.read_parquet(ROOT / 'manifests' / 'stems.parquet')
print('root :', ROOT)
print('songs:', songs.shape, '| stems:', stems.shape)

## 1. Validate the stem-naming / strip rule

The manifest already carries `instrument` (raw, e.g. `피리2`) and `instrument_base` (stripped, `피리`).
**EDA principle: validate a parsing heuristic against the real values before building on it.**
Gugak trap to rule out: numbers that are part of an instrument's *identity* (e.g. 12현 vs 25현 가야금),
not a player index. Below: every raw name with a digit, and the raw→base mapping it produced.

In [ ]:
digit_names = sorted(stems.loc[stems['instrument'].str.contains(r'[0-9]'), 'instrument'].unique())
print('raw instrument names containing a digit:')
print(' ', digit_names)

print('\nraw -> base (only where they differ), with counts:')
diff = stems[stems['instrument'] != stems['instrument_base']]
print(diff.groupby(['instrument', 'instrument_base']).size()
          .sort_values(ascending=False).to_string())

print('\nsongs with >1 stem sharing an instrument_base (true multi-instrument):')
dup = stems.groupby(['song_id', 'instrument_base']).size()
dup = dup[dup > 1].sort_values(ascending=False)
print(f'  {len(dup)} (song, base) groups; genres involved:',
      sorted(stems[stems.song_id.isin(dup.index.get_level_values(0))].genre_sub.unique()))
print(dup.head(10).to_string())

**Result:** every digit is a trailing single digit on a legitimate base instrument (아쟁2, 피리2, …) —
no identity-number trap (no `가야금25`). Multi-instrument songs are **100% 창작국악**. Strip rule is safe.

## 2. Sample rate — confirm the 96 kHz resample set

Everything is 48 kHz except a 창작국악 subset at 96 kHz. Confirm the exact count and that it's
genre-localized, from the manifest headers. (CLAUDE.md: ~130 files → resample to 48 kHz.)

In [ ]:
print('stem sample-rate counts:');   print(stems['sr'].value_counts().to_string())
print('\nmaster sample-rate counts:'); print(songs['master_sr'].value_counts().to_string())

hi = stems[stems['sr'] != 48000]
hi_master = songs[songs['master_sr'] != 48000]
print(f'\n96 kHz: {len(hi)} stems + {len(hi_master)} masters = {len(hi) + len(hi_master)} files'
      f'  across {hi.song_id.nunique()} songs')
print('genres of 96 kHz stems:', sorted(hi.genre_sub.unique()))

## 3. Listen: do 피리 / 피리2 / 피리3 share a timbre?

If they're the same source class (same instrument, different players), we **sum** them into one
target signal — the model can't (and shouldn't) split same-timbre players apart.

**Gotcha we hit and fixed:** gugak stems are *sparse* — an instrument may play only a small fraction
of the song (some here are active <10% of the time). A fixed time window often lands on silence, and
`ipd.Audio` divides by zero when normalizing an all-silent clip. So we **find each stem's loudest
window** and audition there. Note: `ipd.Audio` normalizes each clip, so all players sound equally
loud — compare true levels with the printed `peak`/`rms`, not by ear (levels span ~30 dB across stems).

In [ ]:
def load_active_excerpt(rel_path: str, seconds: float = 15.0, hop_s: float = 0.5):
    """Return the loudest `seconds`-long window of a WAV (float32, stereo), via a sliding-RMS scan.

    One full decode + an O(1)-per-window energy scan (cumulative sum). Fine for a handful of stems.
    """
    audio, sr = sf.read(ROOT / rel_path, dtype='float32', always_2d=True)  # (frames, ch)
    win = min(int(seconds * sr), len(audio))
    mono = audio.mean(axis=1).astype(np.float64)
    csum = np.concatenate([[0.0], np.cumsum(mono ** 2)])         # prefix energy
    hop = max(1, int(hop_s * sr))
    starts = range(0, len(audio) - win + 1, hop)
    energies = np.array([csum[s + win] - csum[s] for s in starts]) if len(audio) > win else np.array([0.0])
    best = list(starts)[int(energies.argmax())] if len(audio) > win else 0
    excerpt = audio[best:best + win]
    win_rms = float(np.sqrt(energies.max() / win)) if len(audio) > win else float(np.sqrt((mono ** 2).mean()))
    return excerpt, sr, best / sr, win_rms

def play(rel_path: str, label: str, seconds: float = 15.0):
    exc, sr, t0, win_rms = load_active_excerpt(rel_path, seconds)
    peak = float(np.abs(exc).max())
    if peak < 1e-4:
        print(f'{label:8}  SILENT everywhere (peak≈0) — no player'); return
    print(f'{label:8}  loudest @ {t0:6.1f}s  win_rms={win_rms:.4f}  peak={peak:.3f}')
    ipd.display(ipd.Audio(exc.T, rate=sr))   # .T -> (ch, frames); normalized per-clip

SONG = '0979_창작국악_창작국악'
piri = stems[(stems.song_id == SONG) & (stems.instrument_base == '피리')].sort_values('instrument')
for _, r in piri.iterrows():
    play(r.stem_path, r.instrument)

For contrast, listen across *different* instruments in the 관악 group (피리 vs 대금 vs 단소)
— distinct timbres the model **should** separate, unlike the 피리 copies above.

In [ ]:
gwan = (stems[(stems.song_id == SONG) & (stems.instrument_base.isin(['피리', '대금', '단소']))]
        .drop_duplicates('instrument_base').sort_values('instrument_base'))
for _, r in gwan.iterrows():
    play(r.stem_path, r.instrument_base)

## 4. Does summing the stems give you the master?  (3 exemplars)

We build training mixtures by **summing stems** (mixture := Σ stems, targets := stems —
self-consistent by construction). Separate question: is the *real master* that same signal? Here we
compare each master directly against its **naive equal-weight stem sum**.

**How to read the panels:**
- **Left** — waveform overlay, ~40 ms of a loud passage. Do master and Σstems trace the same shape?
- **Right** — every audio sample as a dot: **x = Σstems at that instant, y = master at that instant.**
  If master = sum, every dot lands on the diagonal `y = x`. A **tilted straight line** = linear but
  wrong level (a fader/volume fix); a **fuzzy cloud** = the master carries content the sum doesn't
  (reverb/FX). Annotation gives the residual for the naive sum vs the best per-stem linear fit.

Tiers (from the stratified sample — `scripts/residual_test.py`):
- **Tier 1 `0718_민속악_민요`** — master **is** the naive sum (−33 dB).
- **Tier 2 `0151_정악_궁중음악`** — naive sum off (−2.8 dB), but a per-stem **fader** fit recovers it
  (−32 dB) → *linear, wrong levels.*
- **Tier 3 `0885_창작국악`** — even the best linear fit stalls at −15 dB → *nonlinear / added FX.*

Stems are heterophonic → collinear, so the fitted fader weights can be non-physical — trust the
**residual dB**, not the weights.

**Takeaway:** summing stems is safe for *training* (it defines our mixture); the real master is a
**secondary, domain-shifted eval**, not the primary metric — and we don't try to invert master→stems.

In [ ]:
import sys
from matplotlib.colors import LogNorm
sys.path.insert(0, str(ROOT))
from src.data.audio import (read_audio, to_mono, estimate_lag, fit_linear_mix,
                            prefix_energy, loudest_window, rms, to_db)

sp = stems.groupby('song_id')['stem_path'].apply(list).to_dict()
songs_i = songs.set_index('song_id')
EXEMPLARS = [
    ('0718_민속악_민요', 'Tier 1 · master IS the naive sum'),
    ('0151_정악_궁중음악', 'Tier 2 · linear, but needs per-stem faders'),
    ('0885_창작국악_창작국악', 'Tier 3 · master \u2260 any linear stem mix'),
]
C_MASTER, C_SUM = '#0072B2', '#D55E00'   # Okabe-Ito blue / vermillion (CVD-safe)
SR = 48000

fig, axes = plt.subplots(3, 2, figsize=(11, 12), gridspec_kw={'width_ratios': [1.15, 1]})
for row, (sid, title) in enumerate(EXEMPLARS):
    S = [read_audio(ROOT / p)[0].astype('float64') for p in sp[sid]]
    L = min(len(s) for s in S); S = [s[:L] for s in S]
    naive = np.sum(S, axis=0)
    master = read_audio(ROOT / songs_i.loc[sid, 'master_path'])[0].astype('float64')
    d = estimate_lag(master, naive, SR)
    if d >= 0:
        master = master[d:]
    else:
        S = [s[-d:] for s in S]; naive = np.sum(S, axis=0)
    Lc = min(len(master), len(naive)); master = master[:Lc]; naive = naive[:Lc]; S = [s[:Lc] for s in S]

    resid_naive = to_db(rms(master - naive) / rms(master))
    _, recon = fit_linear_mix(S, master)
    resid_ps = to_db(rms(master - recon) / rms(master))
    mm, ms = to_mono(master), to_mono(naive)

    # left: waveform overlay, ~40 ms of a loud passage (short window so shapes are visible)
    win = int(0.040 * SR)
    start = loudest_window(prefix_energy(mm), win, hop=int(0.005 * SR))
    t = np.arange(win) / SR * 1000
    ax = axes[row, 0]
    ax.plot(t, ms[start:start + win], color=C_SUM, lw=1.3, alpha=0.85, label='Σ stems (naive sum)')
    ax.plot(t, mm[start:start + win], color=C_MASTER, lw=1.3, alpha=0.85, label='master')
    ax.set_title(title, fontsize=11, loc='left', weight='bold')
    ax.set_xlabel('ms'); ax.set_ylabel('amplitude')
    ax.spines[['top', 'right']].set_visible(False); ax.grid(True, lw=0.4, alpha=0.3)
    if row == 0:
        ax.legend(frameon=False, fontsize=9, loc='upper right')

    # right: master vs naive sum, sample-by-sample density
    step = max(1, len(mm) // 200_000)
    x, y = ms[::step], mm[::step]
    lim = float(np.percentile(np.abs(np.concatenate([x, y])), 99.9))
    ax = axes[row, 1]
    ax.hexbin(x, y, gridsize=70, cmap='Blues', norm=LogNorm(), extent=(-lim, lim, -lim, lim))
    ax.plot([-lim, lim], [-lim, lim], color='#444', lw=1.0, ls='--', label='y = x  (master = sum)')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
    ax.set_xlabel('Σ stems sample'); ax.set_ylabel('master sample')
    ax.spines[['top', 'right']].set_visible(False)
    ax.text(0.04, 0.96, f'naive sum: {resid_naive:+.1f} dB\nbest linear fit: {resid_ps:+.1f} dB',
            transform=ax.transAxes, va='top', ha='left', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='#ccc', alpha=0.9))
    if row == 0:
        ax.legend(frameon=False, fontsize=8, loc='lower right')

fig.suptitle('Does summing the stems give you the master?  (residual rel. master; lower dB = closer)',
             fontsize=12.5, weight='bold', y=0.997)
fig.tight_layout()
fig.savefig(ROOT / 'notebooks' / 'fig_master_vs_sum.png', dpi=110, bbox_inches='tight')
plt.show()

## 5. Dataset overview (deliverable)

Manifest-only stats (no audio decode) for the instructor briefing: 4-stem class balance,
genre composition, instrument prevalence, and ensemble size. 4-stem groups are color-coded
consistently (관악/찰현/발현/타악). Saves `notebooks/fig_dataset_overview.png`.

Headlines: **4-stem balance is healthy** (관악 31% / 찰현 25% / 발현 25% / 타악 20% by hours);
판소리 (269 songs) and 창작국악 (187) dominate; the "big 6" 대금·피리·해금·가야금·거문고·아쟁 appear in
~600–700 songs each; ensemble size is bimodal (2 = 산조 solo+장구, ~7 = typical, up to 18 = 창작국악).

In [ ]:
import koreanize_matplotlib  # noqa: F401  (registers NanumGothic so Korean labels render)
from matplotlib.patches import Patch

H = lambda s: s.sum() / 3600
GROUP_ORDER = ['관악', '찰현', '발현', '타악']
GC = {'관악': '#0072B2', '찰현': '#D55E00', '발현': '#009E73', '타악': '#E69F00'}  # CVD-safe
ACCENT = '#4C72B0'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# A: 4-stem class balance (hours)
ax = axes[0, 0]
sg = stems.groupby('stem_group_4').dur_s.apply(H).reindex(GROUP_ORDER); tot = sg.sum()
bars = ax.bar(GROUP_ORDER, sg.values, color=[GC[g] for g in GROUP_ORDER], width=0.7)
for b, v in zip(bars, sg.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 1, f'{v:.0f} h\n{v / tot * 100:.0f}%',
            ha='center', va='bottom', fontsize=10)
ax.set_title('4-stem class balance (stem-hours)', fontsize=12, weight='bold', loc='left')
ax.set_ylabel('hours'); ax.set_ylim(0, sg.max() * 1.2)
ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='y', lw=0.4, alpha=0.3)

# B: genre composition (songs, hours annotated)
ax = axes[0, 1]
gsong = songs.genre_sub.value_counts(); ghours = stems.groupby('genre_sub').dur_s.apply(H)
order = gsong.index[::-1]
ax.barh(order, gsong[order].values, color=ACCENT, height=0.7)
for i, g in enumerate(order):
    ax.text(gsong[g] + 3, i, f'{gsong[g]}  ·  {ghours[g]:.0f} h', va='center', fontsize=9)
ax.set_title('Songs per genre (·  stem-hours)', fontsize=12, weight='bold', loc='left')
ax.set_xlabel('songs'); ax.set_xlim(0, gsong.max() * 1.28)
ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='x', lw=0.4, alpha=0.3)

# C: instrument prevalence (top 12), colored by 4-stem group
ax = axes[1, 0]
ib = (stems.groupby('instrument_base')
            .agg(songs=('song_id', 'nunique'), group=('stem_group_4', 'first'))
            .sort_values('songs', ascending=False).head(12).iloc[::-1])
ax.barh(ib.index, ib.songs.values, color=[GC[g] for g in ib.group], height=0.72)
for i, v in enumerate(ib.songs.values):
    ax.text(v + 6, i, str(v), va='center', fontsize=9)
ax.set_title('Instruments by prevalence (top 12, # songs present)', fontsize=12, weight='bold', loc='left')
ax.set_xlabel('songs present'); ax.set_xlim(0, ib.songs.max() * 1.15)
ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='x', lw=0.4, alpha=0.3)
ax.legend(handles=[Patch(color=GC[g], label=g) for g in GROUP_ORDER],
          frameon=False, fontsize=9, ncol=4, loc='lower right')

# D: ensemble size (stems per song)
ax = axes[1, 1]
ns = songs.n_stems
ax.hist(ns, bins=np.arange(0.5, ns.max() + 1.5, 1), color=ACCENT, alpha=0.9, rwidth=0.9)
med = ns.median(); ax.axvline(med, color='#D55E00', lw=1.5, ls='--')
ax.text(med + 0.3, ax.get_ylim()[1] * 0.9, f'median {med:.0f}', color='#D55E00', fontsize=9)
ax.set_title('Ensemble size (stems per song)', fontsize=12, weight='bold', loc='left')
ax.set_xlabel('stems in song'); ax.set_ylabel('songs')
ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='y', lw=0.4, alpha=0.3)

fig.suptitle(f'Gugak dataset overview — {len(songs)} songs · {len(stems)} stems · '
             f'{H(stems.dur_s):.0f} stem-hours · split 721/91/91',
             fontsize=13.5, weight='bold', y=0.998)
fig.tight_layout()
fig.savefig(ROOT / 'notebooks' / 'fig_dataset_overview.png', dpi=110, bbox_inches='tight')
plt.show()

## 6. Cross-dataset instrument distribution (71955 + 71470)

Total **duration (hours) per instrument group, split by dataset** — how much material each of the two
sets contributes per instrument. Instruments are mapped into **71955's original 9 groups**
(대금 · 피리 · 해금 · 거문고 · 가야금 · 아쟁 · 양금 · 타악기 · 기타). 71470 instruments with no home in that
scheme fall into **기타** (vocals, obscure winds/strings like 금·슬·퉁소·나발); 71470 percussion (징·소고·
정주·종 …) → **타악기**. Bars sorted by total hours; caption on each bar = total items (71955 stems +
71470 clips). Saves `figs/instrument_distribution.png` (gitignored — regenerable from this cell).

**Self-contained** — reads audio headers directly from `data/gugak_ensemble_71955/` and
`data/gugak_solo_71470/`, *not* the manifest (which is mid-rebuild), so it runs on its own.

**Headline:** 71955 (~369 stem-h) dominates raw duration; 71470 (~40 clip-h) adds **breadth**, not bulk
— its biggest contribution is 기타 (2,735 vocal clips 71955 never had, ⇒ 3,457 clips there) and a large
boost to melodic *clip counts* (해금 +1,291, 가야금 +1,356). Validation: the 71955 per-group stem counts
here (타악기 1,142 · 피리 762 · 대금 759 …) match the official 71955 group tallies exactly.

In [ ]:
# --- Cross-dataset instrument-group duration (71955 stems + 71470 clips) ---
# Self-contained: reads audio headers straight from disk (the manifest is mid-rebuild).
# Saves figs/instrument_distribution.png. Output need not display in-notebook.
import re, json
from pathlib import Path
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import koreanize_matplotlib  # noqa: F401  (registers NanumGothic so Korean labels render)

def _root(p=None):
    p = Path.cwd() if p is None else p
    for c in [p, *p.parents]:
        if (c / 'pyproject.toml').exists():
            return c
    raise FileNotFoundError('repo root (pyproject.toml) not found')

RT = _root()
D1 = RT / 'data' / 'gugak_ensemble_71955'   # 71955 ensemble (full-song stems)
D2 = RT / 'data' / 'gugak_solo_71470'        # 71470 solo (short clips)
NINE = ['대금', '피리', '해금', '거문고', '가야금', '아쟁', '양금', '타악기', '기타']

# 71955 stem name (trailing digit stripped) -> group. This mapping DEFINES the 9-group scheme;
# every 71955 instrument has a home, so nothing here falls through to 기타 by accident.
_G1 = {'대금': '대금', '피리': '피리', '해금': '해금', '거문고': '거문고', '가야금': '가야금',
       '아쟁': '아쟁', '양금': '양금',
       '장구': '타악기', '소리북': '타악기', '북': '타악기', '좌고': '타악기', '대고': '타악기',
       '바라': '타악기', '꽹과리': '타악기', '편경': '타악기', '방향': '타악기', '박': '타악기',
       '편종': '타악기', '목탁': '타악기', '방울': '타악기',
       '소금': '기타', '단소': '기타', '태평소': '기타', '생황': '기타'}
g1 = lambda inst: _G1.get(re.sub(r'\d+$', '', inst), '기타')

# 71470 instrument_cd -> group: percussion -> 타악기, matching majors -> that major, everything
# else (vocals, obscure winds/strings) -> 기타, per the "map into 71955's 9 groups" decision.
_MAJ = {'SP01': '가야금', 'SP02': '거문고', 'SR01': '해금', 'SR02': '아쟁',
        'WN01': '대금', 'WR01': '피리', 'PT01': '양금'}
def g2(cd):
    if cd.startswith('V'):
        return '기타'
    return _MAJ.get(cd, '타악기' if (cd.startswith('PN') or cd.startswith('PT')) else '기타')

# aggregate: group -> [hours_71955, hours_71470, n_71955, n_71470]
agg = {g: [0.0, 0.0, 0, 0] for g in NINE}
for song in (D1 / 'source').iterdir():
    if not song.is_dir():
        continue
    for w in song.glob('*.wav'):
        inst = w.stem[len(song.name) + 1:]
        if inst == 'master':
            continue
        try:
            info = sf.info(str(w)); dur = info.frames / info.samplerate
        except Exception:
            dur = 0.0
        g = g1(inst); agg[g][0] += dur / 3600; agg[g][2] += 1
for w in (D2 / 'audio').glob('*.wav'):
    try:
        cd = json.loads((D2 / 'labels' / f'{w.stem}.json').read_text())['music_type_info']['instrument_cd']
    except Exception:
        continue
    try:
        info = sf.info(str(w)); dur = info.frames / info.samplerate
    except Exception:
        dur = 0.0
    g = g2(cd); agg[g][1] += dur / 3600; agg[g][3] += 1

# order by total hours (most hours left), split into per-dataset arrays
order = sorted(NINE, key=lambda g: -(agg[g][0] + agg[g][1]))
h1 = np.array([agg[g][0] for g in order]); h2 = np.array([agg[g][1] for g in order])
n1 = np.array([agg[g][2] for g in order]); n2 = np.array([agg[g][3] for g in order])

# stacked bars, 2 CVD-safe colours (Okabe-Ito teal-green + reddish-purple; deliberately not orange/blue)
C_1955, C_1470, INK, MUTED = '#009E73', '#CC79A7', '#222222', '#666666'
x = np.arange(len(order)); top = h1 + h2
fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x, h1, 0.68, color=C_1955, edgecolor='white', linewidth=0.8, label='71955 ensemble (stems)')
ax.bar(x, h2, 0.68, bottom=h1, color=C_1470, edgecolor='white', linewidth=0.8, label='71470 solo (clips)')
for xi, t, a, b in zip(x, top, n1, n2):   # total item count (bold) + per-dataset split (faint)
    ax.text(xi, t + top.max() * 0.012, f'{a + b:,}', ha='center', va='bottom',
            fontsize=10, color=INK, weight='bold')
    ax.text(xi, t + top.max() * 0.055, f'{a:,}+{b:,}', ha='center', va='bottom', fontsize=7.5, color=MUTED)
ax.set_xticks(x); ax.set_xticklabels(order, fontsize=12)
ax.set_ylabel('total duration (hours)', fontsize=11, color=INK); ax.set_ylim(0, top.max() * 1.18)
fig.suptitle('Instrument-group duration by dataset — 71955 ensemble stems + 71470 solo clips',
             x=0.012, ha='left', fontsize=13, weight='bold', color=INK, y=0.99)
ax.set_title('bars sorted by total hours · caption = total items (71955 stems + 71470 clips)',
             loc='left', fontsize=9, color=MUTED, pad=10)
ax.legend(frameon=False, fontsize=10, loc='upper right')
ax.spines[['top', 'right']].set_visible(False); ax.spines[['left', 'bottom']].set_color('#cccccc')
ax.tick_params(colors=MUTED); ax.grid(axis='y', lw=0.4, alpha=0.35); ax.set_axisbelow(True)
fig.tight_layout()
(RT / 'figs').mkdir(exist_ok=True)
fig.savefig(RT / 'figs' / 'instrument_distribution.png', dpi=130, bbox_inches='tight')
plt.close(fig)
print('saved figs/instrument_distribution.png  ·  71955',
      f'{h1.sum():.0f} h/{n1.sum()} stems · 71470 {h2.sum():.0f} h/{n2.sum()} clips')